# HW 3: Homogeneous Transformations
* Continue using the cumulative `byu_robomanip.transforms` module from HW02.
* Complete the HW03-labeled `se3` and `inv` methods.
* Run the provided cells and check each transform direction carefully.

In [ ]:
from byu_robomanip import transforms as tr
import numpy as np
from byu_robomanip.visualization import VizScene
import time

# Problem # 1
### Homogeneous matrix made of only rotation about z-axis by amount $\frac{\pi}{4}$

Rotation by amount theta around the z-axis should give the following:
$$
\left[\begin{matrix}0.707 & -0.707 & 0 & 0\\ 0.707 & 0.707 & 0 & 0\\0 & 0 & 1 & 0\\0 & 0 & 0 & 1\end{matrix}\right]
$$

In [ ]:
T = tr.se3(tr.rotz(np.pi/4))
print(T)

### Translation only along the x-axis by an amount $0.5$ should give the following:
$$
\left[\begin{matrix}1 & 0 & 0 & 0.5 \\0 & 1 & 0 & 0\\0 & 0 & 1 & 0\\0 & 0 & 0 & 1\end{matrix}\right]
$$

In [ ]:
T = tr.se3(p=[0.5, 0, 0])
print(T)

### Checking the "inv" function

If we calculate a homogeneous transform with a rotation of 45 degrees about the x-axis and a general translation ($[0.5, 0.25, 0.3]^T$) we get the following:
$$
\left[\begin{matrix}1 & 0 & 0 & 0.5\\0 & 0.707106781186548 & -0.707106781186547 & 0.25\\0 & 0.707106781186547 & 0.707106781186548 & 0.3\\0 & 0 & 0 & 1\end{matrix}\right]
$$

Then, we can multiply the original transform by its inverse to check out inverse function, we should get the following:
$$
\left[\begin{matrix}1 & 0 & 0 & 0\\0 & 1.0 & 0 & 0\\0 & 0 & 1.0 & 0\\0 & 0 & 0 & 1\end{matrix}\right]
$$

In [ ]:
T = tr.se3(tr.rotx(np.pi/4), [0.5, 0.25, 0.3])
print(T)

# now we can check if we implemented "inv" correctly:
check = tr.inv(T) @ T
print("\n\n inv(T) @ T should give identity matrix:")
print(check)

# Transform a point in one frame to another frame

### Using DH parameters to find A matrices (for each joint):

Remember that if we combine a rotation in z, translation in z, then translation in x, and rotation in x, we should get the same result as the book for following the DH convention to move from one frame (or joint) to another as follows: 

$$
A = \left[\begin{matrix}\cos{\left(\theta \right)} & - \sin{\left(\theta \right)} \cos{\left(\alpha \right)} & \sin{\left(\alpha \right)} \sin{\left(\theta \right)} & a \cos{\left(\theta \right)}\\\sin{\left(\theta \right)} & \cos{\left(\alpha \right)} \cos{\left(\theta \right)} & - \sin{\left(\alpha \right)} \cos{\left(\theta \right)} & a \sin{\left(\theta \right)}\\0 & \sin{\left(\alpha \right)} & \cos{\left(\alpha \right)} & d\\0 & 0 & 0 & 1\end{matrix}\right]
$$

In future homework, we'll implement representations of robot arms that include this transformation. But for the test values of DH parameters below, we can perform this sequence of operations manually.  

Assuming the following DH parameters for two joints:

$\theta_1 = \frac{\pi}{8}$, $d_1 = 0$, $a_1 = 0.3$, $\alpha_1 = \frac{\pi}{2}$

$\theta_2 = \frac{\pi}{4}$, $d_2 = 0$, $a_2 = 0.3$, $\alpha_2 = 0$


The resulting homogeneous transform describing the tip (or frame 2) relate to frame 0 would be:

$$
\left[\begin{matrix}0.653281482438188 & -0.653281482438188 & 0.38268343236509 & 0.473148304484842\\0.270598050073099 & -0.270598050073099 & -0.923879532511287 & 0.195984444731456\\0.707106781186548 & 0.707106781186548 & 0 & 0.212132034355964\\0 & 0 & 0 & 1.0\end{matrix}\right]
$$

In [ ]:
# start by substituting the actual values for R and p and making a new SE3 object
# that describes the transformation from frame 0 to frame 1

# TODO - fill this out
# find the transformation from frame 0 to 1
if "T1_in_0" not in globals():
    T1_in_0 = tr.DHval(np.pi / 8, 0, np.pi / 2, 0.3)

# do the same thing for frame 1 to frame 2
if "T2_in_1" not in globals():
    T2_in_1 = tr.DHval(np.pi / 4, 0, 0, 0.3)

# now we can combine the two to get a transformation that describes frame 2
# relative to frame 0
T2_in_0 = T1_in_0 @ T2_in_1

# printing the result
print(T2_in_0)

# use the "add_frame" function to plot both frames (for joint 1 and joint 2) relative
# to a base or ground frame.

Tw_to_frame1 = np.eye(4)
Tw_to_frame2 = np.eye(4)

viz = VizScene()
viz.add_frame(np.eye(4), label="world", axes_label="w")
# Pass the pre-allocated arrays to retain references
viz.add_frame(Tw_to_frame1, label="joint 1", axes_label="1")
viz.add_frame(Tw_to_frame2, label="joint 2", axes_label="2")

time_to_run = 20
refresh_rate = 60
t = 0
start = time.time()

omega1 = np.pi / 2
omega2 = np.pi / 4

while t < time_to_run:
    t = time.time() - start

    theta1 = omega1 * t
    theta2 = omega2 * t

    T1_in_0 = tr.DHval(theta1, 0, np.pi / 2, 0.3)
    T2_in_1 = tr.DHval(theta2, 0, 0, 0.3)

    # Update the frames in-place ([:]) to preserve PyOpenGL memory formatting
    Tw_to_frame1[:] = T1_in_0
    Tw_to_frame2[:] = T1_in_0 @ T2_in_1
    
    viz.update(As=[np.eye(4), Tw_to_frame1, Tw_to_frame2])
    viz.hold(1 / refresh_rate)

viz.close_viz()